# Experiment runner: E1 benchmark and E3 noisy neighbours
Runs the main experiments of the CS545 GAT project on a Colab GPU (T4).
The project code (`src/`, `scripts/`, `tests/`) is cloned from <https://github.com/YashDolorase/cs545-gat>.
The results are the CSV files that `GAT_Project.ipynb` reads.

| Experiment | Setup | Runs |
|---|---|---|
| E1 | MLP, GCN, GraphSAGE, GAT on Cora and CiteSeer | 4 models × 2 datasets × 10 seeds |
| E3 | Same four models on Cora with 0 / 10 / 25 / 50 / 100 % random edges added | 5 noise levels × 4 models × 10 seeds |

**Reproducibility.** Seeds fix the initialisation, dropout and noisy graphs, but some GPU scatter operations are not deterministic, so a rerun can differ slightly from the committed `results/`. In our rerun, means moved by at most 0.5 points and no conclusion changed.

## Setup

In [ ]:
import os, torch

# Clone the project repository and install PyTorch Geometric (Colab ships PyTorch but not PyG)
PROJECT = "/content/project"
if not os.path.exists(PROJECT):
    !git clone -q https://github.com/YashDolorase/cs545-gat.git {PROJECT}
os.chdir(PROJECT)
!pip -q install torch_geometric

import torch_geometric
print(f"torch {torch.__version__} | PyG {torch_geometric.__version__} | "
      f"device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Results directory. The runner skips any (dataset, model, seed) already in its CSV, so it starts empty
# and every run trains from scratch. Google Drive is used when available, so the results survive a disconnect.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/CS545_GAT/fresh_run"
except Exception:
    OUT = "/content/fresh_run"
os.makedirs(OUT, exist_ok=True)
print("results ->", OUT, "| existing files:", os.listdir(OUT) or "none")

## Sanity tests
* The edge-wise softmax sums to 1 over every node's neighbourhood.
* The from-scratch GAT layer (`src/gat_layer.py`) gives the same outputs as PyG's `GATConv` (concatenated and averaged heads).
* Random-edge injection adds the requested number of edges: new, undirected, with no duplicates or self-loops.
* With the attention vectors set to zero, attention is uniform: the measured entropy equals the uniform entropy.

In [ ]:
os.chdir(PROJECT)  # Colab resets the working directory after a reconnect
!python -m pytest tests -v -W ignore

## E1: Benchmark
Each model uses the hyperparameters from its original paper and is trained full-batch with early stopping on validation accuracy.
Test accuracy comes from the standard public split. Ten seeds give the run-to-run spread that GAT's margin is compared against.
Results → `e1_benchmark.csv`.

In [ ]:
os.chdir(PROJECT)
!python -W ignore scripts/run_experiments.py --exp e1 --seeds 10 --out {OUT}

## E3: Noisy neighbours
Random undirected edges between unconnected node pairs are added to Cora, mostly joining different classes.
Accuracy is recorded at each noise level, along with the attention GAT gives injected edges compared with original ones.
Results → `e3_noise.csv`.

In [ ]:
os.chdir(PROJECT)
!python -W ignore scripts/run_experiments.py --exp e3 --seeds 10 --out {OUT}

## Results summary

In [ ]:
import pandas as pd

# E1: test accuracy (%) as mean ± std over seeds
e1 = pd.read_csv(f"{OUT}/e1_benchmark.csv")
s = e1.groupby(["dataset", "model"])["test_acc"].agg(["mean", "std", "count"])
s["test acc (%)"] = (100 * s["mean"]).round(1).astype(str) + " ± " + (100 * s["std"]).round(1).astype(str)
display(s[["test acc (%)", "count"]].unstack("dataset"))

# E3: mean test accuracy (%) per noise level (rows) and model (columns)
e3 = pd.read_csv(f"{OUT}/e3_noise.csv")
display((100 * e3.pivot_table(index="noise", columns="model", values="test_acc", aggfunc="mean")).round(1))

In [ ]:
# Package the results directory for download
!cd {os.path.dirname(OUT)} && zip -qr /content/fresh_run.zip {os.path.basename(OUT)}
from google.colab import files
files.download("/content/fresh_run.zip")